# 1. Three-statement model

**Question.** Can the income statement, balance sheet, and cash flow operate as one system?

**Output of this notebook.** A linked forecast that ties (identity gaps ≈ 0), with exhibits for the revenue build, balance sheet, working capital, and cash rollforward.

Interest starts on beginning debt. A later cell switches to average-debt interest and solves the circularity.


In [ ]:
%matplotlib inline
import sys
from pathlib import Path

def _project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "valuation" / "__init__.py").exists():
            return candidate
    raise RuntimeError("Open this notebook from the project directory.")

ROOT = _project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dataclasses import replace
import matplotlib.pyplot as plt
import pandas as pd

from valuation import boot
from valuation import plots as viz
from valuation import report as R
from valuation.case import TOL
from valuation.statements import Line, audit, combine, forecast_statements, identity_failures, max_abs_gap

# Compact equity picker — changes cases/active.yaml for EVERY notebook.
case, UNIT = boot(picker=True)


## 1. Historical statements


In [ ]:
R.section("Historical statements", "Sample history is generated by the same engine as the forecast, so it ties.")
R.table(case.history.income, digits=1, caption="Income statement")
R.table(case.history.balance, digits=1, caption="Balance sheet")
R.table(case.history.cashflow, digits=1, caption="Cash flow statement")
viz.plot_income_stack(case.history, UNIT)
plt.show()
viz.plot_balance_composition(case.history, UNIT)
plt.show()
viz.plot_working_capital(case.history, UNIT)
plt.show()


## 2. Forecast and identity check


In [ ]:
from valuation.statements import DAYS

last = case.history.income_years()[-1]
forecast = forecast_statements(case, interest="beginning")
full = combine(case.history, forecast.statements)
R.show_statements_headline(case, forecast)

hist_end = last
viz.plot_income_stack(full, UNIT, history_end=hist_end)
plt.show()
viz.plot_balance_composition(full, UNIT, history_end=hist_end)
plt.show()
viz.plot_cash_bridge(forecast.statements.cashflow, UNIT)
plt.show()

R.table(forecast.statements.income, digits=1, caption="Forecast income statement")
R.table(forecast.statements.balance, digits=1, caption="Forecast balance sheet (includes opening year)")
R.table(forecast.statements.cashflow, digits=1, caption="Forecast cash flow")


In [ ]:
hist_gaps = audit(case.history)
fc_gaps = audit(forecast.statements)
R.kpi_row([
    ("Hist. max |gap|", f"{max_abs_gap(hist_gaps):,.4f}"),
    ("Forecast max |gap|", f"{max_abs_gap(fc_gaps):,.8f}"),
    ("Forecast ties", "Yes" if identity_failures(fc_gaps, tol=TOL).empty else "No"),
])
assert identity_failures(fc_gaps, tol=TOL).empty
if case.synthetic:
    assert identity_failures(hist_gaps, tol=TOL).empty
    R.note("Sample history and forecast both tie within tolerance.")
else:
    fails = identity_failures(hist_gaps, tol=1.0)
    if fails.empty:
        R.note("Live history: no gaps above $1m on mapped lines.")
    else:
        R.table(fails, digits=1, caption="Live history gaps (unmapped filing items)")
    viz.plot_audit_heatmap(hist_gaps)
    plt.show()


## 3. Average-debt interest


In [ ]:
average = forecast_statements(case, interest="average")
R.kpi_row([
    ("Converged", str(average.converged)),
    ("Iterations", str(average.iterations)),
])
R.table(pd.DataFrame({
    "Beginning-debt interest": forecast.statements.income.loc[Line.INTEREST],
    "Average-debt interest": average.statements.income.loc[Line.INTEREST],
}), digits=2, caption="Interest expense under both conventions")
assert average.converged
assert identity_failures(audit(average.statements), tol=TOL).empty
